In [1]:
# -------- Imports --------
import sys
import os
import numpy as np
import scipy.sparse as sp
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from _utility import *
from _fractures import *
from _initial_conditions import *

# -- Qiskit imports --
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import MatrixExponential
from qiskit.quantum_info import SparsePauliOp, Pauli, Operator
from qiskit.primitives import StatevectorEstimator

np.random.seed(0)


In [3]:
# -------- Parameters --------
# -- Grid parameters --

xmin,ymin,zmin,xmax,ymax,zmax,Nx,Ny,Nz,dx,dy,dz = get_grid_parameters()

N_main,N_vx,N_vy,N_vz,N_sxy,N_sxz,N_syz,N_vel,N_stress,psi_len = get_grid_size(Nx,Ny,Nz)

In [5]:
#Define a function to get the indices of the points for the subspace energy calculation
def get_mask_indices(ith_element,wave_field_component):
    subspace_points = []
    for i in range(len(ith_element)):
        for j in ith_element[i]: 
            if(wave_field_component=='vx'):
                subspace_points.append(j)
            elif(wave_field_component=='vy'):
                subspace_points.append(N_vx+j)
            elif(wave_field_component=='vz'):
                subspace_points.append(N_vx+N_vy+j)
            elif(wave_field_component=='sxx'):
                subspace_points.append(N_vel+j)
            elif(wave_field_component=='syy'):
                subspace_points.append(N_vel+N_main+j)
            elif(wave_field_component=='szz'):
                subspace_points.append(N_vel+2*N_main+j)
            elif(wave_field_component=='sxy'): 
                subspace_points.append(N_vel+3*N_main+j)
            elif(wave_field_component=='sxz'):
                subspace_points.append(N_vel+3*N_main+N_sxy+j)
            elif(wave_field_component=='syz'):
                subspace_points.append(N_vel+3*N_main+N_sxy+N_sxz+j)
            else:
                print("Not a valid wave field component")
                
    return list(filter(lambda x: x < psi_len, subspace_points))


def get_i_location(Nx,Ny,Nz,z_min,z_max,wave_field_component):
    assert z_min >= 0 and z_max <= Nz-1 and z_min < z_max
    i_location = []
    #Should be 7 different unique amounts of grid points for the same z location, the main grid, the points for stress_xy, vx, and vy
    #vx
    if(wave_field_component=='vx'):
        i_location.append(np.arange((Nx-1)*Ny*z_min,(Nx-1)*Ny*z_max,1))
    #vy
    elif(wave_field_component=='vy'):
        i_location.append(np.arange(Nx*(Ny-1)*z_min,Nx*(Ny-1)*z_max,1))
    #vz
    elif(wave_field_component=='vz'):
        i_location.append(np.arange(Nx*Ny*z_min,Nx*Ny*(z_max),1))
    #main
    elif(wave_field_component=='sxx' or wave_field_component=='syy' or wave_field_component=='szz' ):
        i_location.append(np.arange(Nx*Ny*z_min,Nx*Ny*z_max,1))
    #stress_xy
    elif(wave_field_component=='sxy'):
        i_location.append(np.arange((Nx-1)*(Ny-1)*z_min,(Nx-1)*(Ny-1)*z_max,1))
    #stress_xz
    elif(wave_field_component=='sxz'):
        i_location.append(np.arange((Nx-1)*(Ny)*z_min,(Nx-1)*Ny*(z_max),1))
    #stress_yz
    elif(wave_field_component=='syz'):
        i_location.append(np.arange(Nx*(Ny-1)*z_min,Nx*(Ny-1)*(z_max),1))
    else:
        print('Not valid wave field component')

    print(f'i_location {type(i_location)}: {i_location}')
    return i_location

In [7]:
# A function returning a dictionary of masks for kinetic, potential, and total energy subspace projectors
def threeProjectors(mask, Nx, Ny, Nz):
    kinetic = np.copy(mask)
    potential = np.copy(mask)
    kinetic_num = (Nx-1)*Ny*Nz + Nx*(Ny-1)*Nz + Nx*Ny*(Nz-1)
    kinetic[kinetic_num:] = 0
    potential[:kinetic_num] = 0
    return {
        'kinetic' : kinetic,
        'potential' : potential,
        'total' : mask
    }

#mask_dict = threeProjectors(mask, Nx, Ny, Nz)
#print(mask_dict)

In [9]:
# -------- Simulation --------
# Simulate the quantum circuit
def hamiltonian_simulation(qc,observable):
    estimator = StatevectorEstimator()
    job = estimator.run([(qc, observable)], precision=1e-8)
    result = job.result()
    print(f" Layer > Expectation value: {result[0].data.evs}")
    print(f" > Metadata: {result[0].metadata}")
    return result


In [11]:
# Important: The following implementation of the material parameters is not scalable. This code is only for demonstration purposes and assumes oracle access. 
# In a real-world scenario, the material parameters and the Hamiltonian need to be sparsly constructed on the QC itself.
# Furthermore, this code uses direct matrix exponentiation, which is inefficient for large matrices. Other integration methods should be used (e.g. Trotter-Suzuki, Qubitization).


# Density model and S matrix (3D with one horizontal fracture

#Set fracture thickness
fracture_thickness = 0
rho_model, S_matrix = one_horizontal_fracture(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)

# -------- Simulation (3D Elastic) (Oracle Access) --------
(H, A, _, B_sqrt, B_inv, _) = FD_solver_3D_elastic_quantum(Nx, Ny, Nz, dx, dy, dz, rho_model, S_matrix)

H_pauli = SparsePauliOp.from_operator(Operator(H.toarray())) # Expensive conversion
print("Hamiltonian shape: ", H.shape)
print("Hamiltonian NNZ-Ratio: ", H.nnz/H.shape[0]**2)
print("Pauli Terms (inefficient representation): ", len(H_pauli))

# Number of grid points
N = Nx*Ny*Nz
print("Number of grid points: ", N)

# Number of qubits
n = (H.shape[0]-1).bit_length()
print("Number of qubits (for wave field): ", n)




Adding horizontal fracture planes:
  Fracture: Horizontal plane at z-index 4
  Total fracture cells: 16 out of 729 (2.2%)

Density Model Statistics:
  Min: 1000.0 kg/m³, Max: 2700.0 kg/m³
  Mean: 2662.7 kg/m³, Std: 249.1 kg/m³
Hamiltonian shape:  (8192, 8192)
Hamiltonian NNZ-Ratio:  0.0003218650817871094
Pauli Terms (inefficient representation):  2697336
Number of grid points:  729
Number of qubits (for wave field):  13


In [12]:
# -- Initial Conditions (3D Elastic) --
# State vector: [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

phi_0 = gaussian_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)

# Pad the initial conditions with zeros to match Hamiltonian size (power of 2)
phi_0 = np.concatenate([phi_0, np.zeros(H.shape[0] - psi_len)])

# Normalize the initial state and transform it to the energy basis
psi_0 = B_sqrt @ phi_0
norm = np.linalg.norm(psi_0)
psi_0 /= norm

# Number of non-zero initial state values
psi_0_nnz = np.sum(psi_0 != 0)
print('Initial state NNZ-Ratio:', psi_0_nnz/psi_len)

# Pad the mask with zeros
#mask = np.concatenate([mask, np.zeros(H.shape[0]-psi_len)])
#print(len(mask))

Initial state NNZ-Ratio: 0.37327188940092165


In [14]:
# -------- Quantum Circuit --------
# -- Time-evolution Synthesis --
# Note:H = 1j  * H_real where H_real is real and symmetric (hence Hermitian)
# The classical evolution is: |ψ(t)⟩ = exp(-1j * H * t) |ψ(0)⟩ = exp(H_real * t) |ψ(0)⟩
# PauliEvolutionGate implements exp(-i * H_pauli * t) where H_pauli must be Hermitian with real coefficients
# Since H_pauli = H_real, we get exp(-i * H_real * t) = cos(H_real * t) - i*sin(H_real * t)
# For small times or when H_real has small eigenvalues, the real part approximates exp(H_real * t)
# We use H_real with time t to get unitary evolution exp(-i * H_real * t)
def get_quantum_circuit(mask,t):
    synthesis = MatrixExponential()
    evo = PauliEvolutionGate(H_pauli, time=t, synthesis=synthesis, label='U=exp(-iH_real*t)')

    # -- Quantum Simulation --
    # Define the quantum registers
    reg_wave = QuantumRegister(n, 'wave')
    reg_P = QuantumRegister(1, 'P')

    # Initialize the quantum circuit
    qc = QuantumCircuit(reg_wave, reg_P)
    qc.prepare_state(psi_0, reg_wave)
    print(np.linalg.norm(psi_0))
    # Apply the time-evolution operator
    qc.append(evo, reg_wave)

    # Apply unitary "projection" transform (Can often be combined into fewer MCX gates)
    qc.x(reg_P)
    for d in np.nonzero(mask)[0]:
        qc.mcx(reg_wave, target_qubit=reg_P, ctrl_state=int(d))

    # Define observable
    O1 = Pauli('I'*(n+1))
    O2 = Pauli('Z'+'I'*n) 
    observable = SparsePauliOp([O1, O2], [0.5, 0.5])

    return qc,observable
    #print("Observable Pauli terms: ", len(observable))

    # Draw the circuit
    #qc.draw('mpl')




In [16]:
# Compute the energy loss (post-processing)
def get_quantum_subspace_energy(result):
    E = result[0].data.evs
    EN_QC = (1/2) * E * norm**2 * (dx * dy * dz)
    print('Expected value: ',E)
    print('Subspace Energy Estimate (quantum):', EN_QC.round(30))
    return EN_QC.round(30)


In [17]:
# Time Integration (Has classical integration errors (DOP853))
# Note: Use normalized initial state for fair comparison with quantum
def get_classical_state_vector(t):
    phi_0_normalized = phi_0 / norm  # Normalize using the same norm as quantum
    phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0_normalized, t_eval=(0,t), method='DOP853').y.T[-1] 
    psi_t_classical = B_sqrt @ phi_t
    return psi_t_classical


In [18]:

def get_classical_subspace_energy(mask,psi_t_classical):
    # Calculate subspace energy using the same normalization as quantum
    # The mask selects which states to measure, then we compute the probability
    masked_state = np.diag(mask) @ psi_t_classical
    # Probability of being in the masked subspace
    prob_in_subspace = np.linalg.norm(masked_state)**2 / np.linalg.norm(psi_t_classical)**2
    print("prob in subspace",prob_in_subspace)
    print("norm = ",norm)
    print("dx = ",dx)
    EN_CL = (1/2) * prob_in_subspace * np.linalg.norm(psi_t_classical)**2 * (dx * dy * dz)
    print('Subspace Energy Estimate (classical):', EN_CL.round(30))
    return EN_CL.round(30)

In [26]:
## RUN THE SIMULATION FOR MULTIPLE LAYERS AND TIMES
num_layers = 8
num_times = 2
t = 1
layer_spacing = (Nz-1)//num_layers
times = np.linspace(1/num_times,t,num_times)
layer_min = np.arange(0,Nz-1,layer_spacing)
layer_max = layer_min+layer_spacing

subspace_energies = []

masks = []
mask_layer = []
components = ['vx','vy','vz','sxx','syy','szz','sxy','sxz','syz']
for i in range(num_layers):
    mask_layer=[]
    for j in range(len(components)):
        mask = subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min[i],layer_max[i],components[j]),components[j]))
        mask = np.concatenate([mask, np.zeros(H.shape[0]-psi_len)])
        mask_layer.append(mask)
    masks.append(mask_layer)

i_location <class 'list'>: [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71])]
i_location <class 'list'>: [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71])]
i_location <class 'list'>: [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59

9

In [23]:
## RUN THE SIMULATION FOR MULTIPLE TIMES

test_subspace_energies_quantum = []
test_subspace_energies_classical = []

test_mask = masks[0] #Contains 9 masks for each component
qe_layer = 0
ce_layer = 0
#for m in masks:
for time in range(len(times)):
    qe_layer=0
    ce_layer=0
    for j in range(len(test_mask)):
        print("Computing Mask ",j,"for Layer 1")
        qc, observable = get_quantum_circuit(test_mask[j],times[time])
        result = hamiltonian_simulation(qc,observable)
        qe_layer+=get_quantum_subspace_energy(result)
        psi_t_classical = get_classical_state_vector(times[time])
        ce_layer+=get_classical_subspace_energy(test_mask[j],psi_t_classical)
        print("Quantum Energy: ",qe_layer)
        print("Classical Energy: ",ce_layer)

    test_subspace_energies_quantum.append(qe_layer)
    test_subspace_energies_classical.append(ce_layer)

                                            
    
    
    

NameError: name 'masks' is not defined

[0.5 1. ]


prob in subspace 2.2078809562230674e-15
norm =  3.5852661020596925
dx =  1000.0
Subspace Energy Estimate (classical): (1.103940478111534e-06+0j)


Expected value:  -7.516028151617725e-09
Subspace Energy Estimate (quantum): -48.30601283116877
-48.30601283116877


8192

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1.])

1.0
1.0


1.0
1.0
True


 Layer > Expectation value: 0.5003563260536955
 > Metadata: {'target_precision': 1e-08, 'circuit_metadata': {}}


3215823386.8913817
